This notebook shows how the lower top_X mids files can be derived from the top_50 filenames 
and the HDF5 file containing the data.
Since the lower numbers are subsets of top_50, the filenames can be filtered and matching 
labels need to be generated. The log-melspectrograms are all already contained.

In [1]:
import h5py
import torch
import numpy as np

import utils.utility_funcs as my_funcs

In [2]:
# Paths to the pckl dicts and the valid filenames for 30 mids
PATH_TO_AS_EVAL_30_LABELS = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\audioset_eval_labels_30_dict.pckl'
PATH_TO_AS_TRAIN_30_LABELS = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\audioset_train_labels_30_dict.pckl'

PATH_TO_AS_FNAMES_EVAL_30 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\audioset_eval_filenames_top30.txt'
PATH_TO_AS_FNAMES_TRAIN_30 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\audioset_train_filenames_top30.txt'

In [3]:
valid_filenames_as_train = set()
with open(PATH_TO_AS_FNAMES_TRAIN_30, 'r') as fnames:
    for line in fnames.readlines():
        valid_filenames_as_train.add(line.rstrip('\n'))

valid_filenames_as_eval = set()
with open(PATH_TO_AS_FNAMES_EVAL_30, 'r') as fnames:
    for line in fnames.readlines():
        valid_filenames_as_eval.add(line.rstrip('\n'))
valid_filenames_as_eval

{'YqaCD3TSTn2k',
 'Y-HWygXWSNRA',
 'YAY2h_JWPtao',
 'YbUZQeCVNzKc',
 'YDNBpRJMEm4s',
 'YfIer0uCLqSM',
 'YmZuhtrwOS7Q',
 'Y75wvnWAqoM0',
 'YcADT8fUucLQ',
 'YYkifafWURG8',
 'YYWJJpKjyp04',
 'YdEc7tV30KJo',
 'Y0QqQ1j1vBgw',
 'YCkv52MHdAeg',
 'YK1DHGdAOBH8',
 'YkR4UTeYw44o',
 'Y0KCVgexi4yU',
 'Y0wM_qR6Yihk',
 'Y6Ig6zwBn4b4',
 'YrPcBRxCuCyc',
 'YdOKfGfiqJns',
 'YLIU7qUVBSWU',
 'YRgyqhpOJFM4',
 'YguoU_cuR8EE',
 'YYFhfCrHUL6U',
 'YrSZ86H06ABo',
 'YME7TYB3B27o',
 'Y_gG0KNGD47M',
 'Yu1ru8jm2iDY',
 'YQJumqm9_jz4',
 'YVPhcuvPn6L4',
 'YNymjfq2kXnI',
 'YWhNun_U3cRU',
 'YqW4kBJsudLI',
 'Y6-RIiM19Be8',
 'YDmy4EjohxxU',
 'YbH0ccKXoLZU',
 'YYP5NMmYsHt0',
 'YbrfHK1WL1II',
 'Ybf_8fM2EqgE',
 'YcdbYsoEasio',
 'Y3I3HZLx23E0',
 'YkQUkn0oL8HQ',
 'Y_mGrwBl_BMw',
 'YY0cDPFkhkHU',
 'YLrhy2auO6hY',
 'YuTS8AoBBvIQ',
 'Y6FUmapkqPn0',
 'YMl-jhHJA7s0',
 'YWOZSaBgQVTU',
 'YJErx-AImWqk',
 'YB32m9Jqqrpo',
 'YlVNmt67p86k',
 'YZW7eH1DORJg',
 'YHVy_lIAzhMA',
 'Y9rXAmXuAOw8',
 'Yps8ulH55E5Q',
 'YIfoVNe1IcAA',
 'YJiBAkAwK0GM

In [4]:
# Get dicts
audioset_train_labels_30 = my_funcs.pickle_load(PATH_TO_AS_TRAIN_30_LABELS)
audioset_eval_labels_30 = my_funcs.pickle_load(PATH_TO_AS_EVAL_30_LABELS)
audioset_eval_labels_30

{'s9d-2nhuJCQ': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0],
 'YxlGt805lTA': [0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 '-HAoExLVs6Q': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 '-ByoSbgzr4M': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 '2VjbDttFK44': [0,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'hvBFjdYnW4U': [0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0],
 'eX

In [28]:
# Open the HDF5 file
PATH_TO_AS_DATA_HDF5 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\Audioset_data_cl.hdf5'

train_counter = 0
eval_counter = 0
with h5py.File(PATH_TO_AS_DATA_HDF5, 'a') as as_hdf5:
    train_50_grp = as_hdf5['audioset_strong_train_50']
    eval_50_grp = as_hdf5['audioset_strong_eval_50']

    train_30_grp = as_hdf5.create_group('audioset_strong_train_30')
    eval_30_grp = as_hdf5.create_group('audioset_strong_eval_30')

    print(train_50_grp)
    print(eval_50_grp)
    for fname in train_50_grp.keys():
        fname_wo_Y = fname[1::] # Remove leading Y
        if fname in valid_filenames_as_train:
            # Need to save as np to create an independent dataset
            # Using an existing dataset creates a soft link -> labels are linked as well
            train_30_grp[fname] = np.array(train_50_grp[fname])
            train_30_grp[fname].attrs['label'] = audioset_train_labels_30[fname_wo_Y]
            train_counter += 1
    
    for fname_eval in eval_50_grp.keys():
        fname_eval_wo_Y = fname_eval[1::]
        if fname_eval in valid_filenames_as_eval:
            eval_30_grp[fname_eval] = np.array(eval_50_grp[fname_eval])
            eval_30_grp[fname_eval].attrs['label'] = audioset_eval_labels_30[fname_eval_wo_Y]
            eval_counter += 1

print(f"Found train set files: {train_counter}")
print(f"Found eval set files: {eval_counter}")

# Read through the desired group and the dataset names inside
# Compare names to the valid filename list and copy to a new group
# without label info and insert appropriate label info

<HDF5 group "/audioset_strong_train_50" (81070 members)>
<HDF5 group "/audioset_strong_eval_50" (14242 members)>
Found train set files: 79689
Found eval set files: 13964


In [31]:
counter = 0
with h5py.File(PATH_TO_AS_DATA_HDF5, 'a') as test:
    print(test.keys())
    grp_30 = test['audioset_strong_train_30']
    grp_50 = test['audioset_strong_train_50']
    eval_30 = test['audioset_strong_eval_30']
    eval_50 = test['audioset_strong_eval_50']
    for fname in grp_30.keys():
        print(grp_30[fname])
        print(grp_30[fname].attrs['label'])
        print(grp_50[fname])
        print(grp_50[fname].attrs['label'])
        
        if counter == 1:
            break
        counter += 1

    counter = 0
    for fname in eval_30.keys():
        print(eval_30[fname])
        print(eval_30[fname].attrs['label'])
        print(eval_50[fname])
        print(eval_50[fname].attrs['label'])
    
        if counter == 1:
            break
        counter += 1

    #del test['audioset_strong_train_30']

<KeysViewHDF5 ['audioset_strong_eval_30', 'audioset_strong_eval_50', 'audioset_strong_train_30', 'audioset_strong_train_50']>
<HDF5 dataset "Y--24LG2mr-Y": shape (1, 1001, 64), type "<f4">
[0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
<HDF5 dataset "Y--24LG2mr-Y": shape (1, 1001, 64), type "<f4">
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0]
<HDF5 dataset "Y--5OkAjCI7g": shape (1, 1001, 64), type "<f4">
[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1]
<HDF5 dataset "Y--5OkAjCI7g": shape (1, 1001, 64), type "<f4">
[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 1 1 0 0 1 0]
<HDF5 dataset "Y--BfvyPmVMo": shape (1, 1001, 64), type "<f4">
[0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
<HDF5 dataset "Y--BfvyPmVMo": shape (1, 1001, 64), type "<f4">
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0]
<